# Rotation validation: 360-degree overlap and stitching

Estimate overlap and overlap side for I12 dataset 5 and I13 datasets 1 and 3.
Apply dark/flat correction, then compare the estimates and stitched sinograms
with numerical and structural references. No negative logarithm is applied.

## Data and execution

Use the named dataset archives from [Zenodo record 17252190](https://zenodo.org/records/17252190).
The repository downloader verifies the archive checksums.
Run from the repository root:

```bash
python .scripts/download_zenodo.py Zenodo
python -m pip install -e '.[validation]'
python -m pytest --nbmake --nbmake-timeout=1800 --overwrite docs/source/validation/rotation/center_360.ipynb
```

Use an NVIDIA GPU with compatible CuPy and ToMoBAR installed, plus ASTRA for
ASTRA-based reconstruction. Run notebooks serially. Set `HTTOMOLIBGPU_DATA_DIR`
to use an archive stored elsewhere.

Documentation builds render saved outputs without running GPU computations.
Execute with `--overwrite` before building documentation to update the results.


In [ ]:
import gc
import hashlib
import json
import os
import platform
from importlib.metadata import version
from pathlib import Path

import cupy as cp
import matplotlib.pyplot as plt
import numpy as np

from httomolibgpu.prep.normalize import dark_flat_field_correction, minus_log
from httomolibgpu.recon.rotation import find_center_vo, find_center_pc, find_center_360

repo_root = next(
    (
        p
        for p in (Path.cwd(), *Path.cwd().parents)
        if (p / "pyproject.toml").is_file() and (p / "httomolibgpu").is_dir()
    ),
    None,
)
configured_dir = os.environ.get("HTTOMOLIBGPU_DATA_DIR")
if configured_dir:
    data_dir = Path(configured_dir).expanduser()
elif repo_root is not None:
    data_dir = repo_root / "Zenodo"
else:
    raise FileNotFoundError("Set HTTOMOLIBGPU_DATA_DIR to the Zenodo archive directory")

props = cp.cuda.runtime.getDeviceProperties(cp.cuda.Device().id)
environment = {
    "python": platform.python_version(),
    "packages": {
        name: version(name) for name in ("httomolibgpu", "numpy", "matplotlib")
    },
    "cupy": cp.__version__,
    "cuda_runtime": cp.cuda.runtime.runtimeGetVersion(),
    "cuda_driver": cp.cuda.runtime.driverGetVersion(),
    "gpu": (
        props["name"].decode() if isinstance(props["name"], bytes) else props["name"]
    ),
}
print(json.dumps(environment, indent=2))
results = []
dataset_provenance = {}


def clean_memory():
    gc.collect()
    cp.get_default_memory_pool().free_all_blocks()
    cp.get_default_pinned_memory_pool().free_all_blocks()


def archive_path(name):
    path = data_dir / (name + ".npz")
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing {path}; run .scripts/download_zenodo.py first"
        )
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    dataset_provenance[name] = {
        "archive": path.name,
        "sha256": digest.hexdigest(),
        "zenodo_record": "17252190",
    }
    print(json.dumps(dataset_provenance[name], indent=2))
    return path


def load_transmission(name):
    with np.load(archive_path(name)) as archive:
        raw = cp.asarray(archive["projdata"])
        flats = cp.asarray(archive["flats"])
        darks = cp.asarray(archive["darks"])
        print("Raw, flats, darks:", raw.shape, flats.shape, darks.shape)
    transmission = dark_flat_field_correction(raw, flats, darks)
    del raw, flats, darks
    clean_memory()
    return transmission


def display_sinogram(sinogram, observed, expected, title):
    image = cp.asnumpy(sinogram)
    lo, hi = np.percentile(image[np.isfinite(image)], [1, 99])
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    im = axes[0].imshow(image, cmap="gray", aspect="auto", vmin=lo, vmax=hi)
    axes[0].axvline(float(observed), color="tab:orange", label="Observed centre")
    axes[0].axvline(
        float(expected), color="tab:cyan", linestyle="--", label="Reference centre"
    )
    axes[0].set(
        title=title, xlabel="Detector column (pixel)", ylabel="Projection index"
    )
    axes[0].legend(fontsize=8)
    fig.colorbar(im, ax=axes[0], label="Corrected transmission")
    axes[1].plot(np.mean(image, axis=0))
    axes[1].axvline(float(observed), color="tab:orange")
    axes[1].axvline(float(expected), color="tab:cyan", linestyle="--")
    axes[1].set(
        title="Angular mean profile",
        xlabel="Detector column (pixel)",
        ylabel="Mean transmission",
    )
    plt.show()
    plt.close(fig)


def check_results():
    print(json.dumps(results, indent=2))
    failed = [row for row in results if not row["passed"]]
    assert not failed, "Rotation validation checks failed: " + json.dumps(
        failed, indent=2
    )
    print(f"All {len(results)} rotation validation checks passed.")

## Preprocessing and overlap estimates

Apply dark/flat correction to arrays in `(angle, detector row, detector column)`
order. The first value returned by `find_center_360` is the input detector width,
not an independent rotation-axis coordinate. Check this value alongside the
estimated overlap and its side.

Show the first angular half, the flipped opposing half and a stitched sinogram.
The preview uses the estimated overlap. Full-stack shape, float32 dtype and
contiguity checks use the fixed fractional overlaps shown in each case.
`sino_360_to_180` rounds overlap to an even pixel count and discards an unmatched
final projection. Use angle metadata to handle repeated scan endpoints.


In [ ]:
from httomolibgpu.misc.morph import sino_360_to_180

## I12 DATASET5

Parameters: `{}`.
Check: `int(cor) == 2560`, `side == "left"` and
`int(overlap) == 186`. The integer comparison does not
require a fractional overlap to equal the integer reference exactly.


In [ ]:
data = load_transmission("i12_dataset5")
parameters = {}
cor, overlap, side, overlap_position = find_center_360(data, **parameters)
passed = bool(int(cor) == 2560 and side == "left" and int(overlap) == 186)
results.append(
    {
        "test": "test_center_360_i12_dataset5",
        "dataset": "i12_dataset5",
        "parameters": parameters,
        "returned_center": float(cor),
        "expected_int_center": 2560,
        "overlap": float(overlap),
        "expected_int_overlap": 186,
        "side": side,
        "expected_side": "left",
        "overlap_position": float(overlap_position),
        "passed": passed,
    }
)
print(json.dumps(results[-1], indent=2))
row = parameters.get("ind", data.shape[1] // 2)
sinogram = data[:, row, :]
half_count = sinogram.shape[0] // 2 + 1
opposing = cp.fliplr(sinogram[-half_count:, :])
stitched = sino_360_to_180(data[:, row : row + 1, :], overlap=overlap, side=side)
effective_overlap = int(np.rint(overlap))
if effective_overlap % 2:
    effective_overlap += 1
print(
    "Estimated overlap:",
    float(overlap),
    "effective even-pixel overlap:",
    effective_overlap,
)
print("Input and stitched shapes:", data.shape, stitched.shape)
preview = [
    cp.asnumpy(sinogram[:half_count]),
    cp.asnumpy(opposing),
    cp.asnumpy(stitched[:, 0, :]),
]
lo, hi = np.percentile(preview[0][np.isfinite(preview[0])], [1, 99])
fig, axes = plt.subplots(3, 1, figsize=(12, 9), constrained_layout=True)
for ax, image, title in zip(
    axes,
    preview,
    (
        "First angular half",
        "Opposing half, detector columns reversed",
        "Stitched 180-degree sinogram",
    ),
):
    ax.imshow(image, cmap="gray", vmin=lo, vmax=hi, aspect="auto")
    ax.set(title=title, xlabel="Detector column (pixel)", ylabel="Projection index")
fig.suptitle("i12_dataset5: overlap and side from find_center_360")
plt.show()
plt.close(fig)
# Check full-stack stitching with the fixed reference overlap.
del stitched
clean_memory()
stitched_full = cp.asnumpy(sino_360_to_180(data, overlap=186.66, side="left"))
stitch_passed = (
    stitched_full.shape == (1800, 15, 5120)
    and stitched_full.dtype == np.float32
    and stitched_full.flags.c_contiguous
)
results.append(
    {
        "test": "test_sino_360_to_180_i12_dataset5",
        "overlap": 186.66,
        "side": "left",
        "shape": list(stitched_full.shape),
        "expected_shape": [1800, 15, 5120],
        "dtype": str(stitched_full.dtype),
        "c_contiguous": bool(stitched_full.flags.c_contiguous),
        "passed": bool(stitch_passed),
    }
)
print(json.dumps(results[-1], indent=2))
del stitched_full
del data, sinogram, opposing, preview, image
clean_memory()

## I13 DATASET1

Parameters: `{}`.
Check: `int(cor) == 2560`, `side == "right"` and
`int(overlap) == 473`. The integer comparison does not
require a fractional overlap to equal the integer reference exactly.


In [ ]:
data = load_transmission("i13_dataset1")
parameters = {}
cor, overlap, side, overlap_position = find_center_360(data, **parameters)
passed = bool(int(cor) == 2560 and side == "right" and int(overlap) == 473)
results.append(
    {
        "test": "test_center_360_i13_dataset1",
        "dataset": "i13_dataset1",
        "parameters": parameters,
        "returned_center": float(cor),
        "expected_int_center": 2560,
        "overlap": float(overlap),
        "expected_int_overlap": 473,
        "side": side,
        "expected_side": "right",
        "overlap_position": float(overlap_position),
        "passed": passed,
    }
)
print(json.dumps(results[-1], indent=2))
row = parameters.get("ind", data.shape[1] // 2)
sinogram = data[:, row, :]
half_count = sinogram.shape[0] // 2 + 1
opposing = cp.fliplr(sinogram[-half_count:, :])
stitched = sino_360_to_180(data[:, row : row + 1, :], overlap=overlap, side=side)
effective_overlap = int(np.rint(overlap))
if effective_overlap % 2:
    effective_overlap += 1
print(
    "Estimated overlap:",
    float(overlap),
    "effective even-pixel overlap:",
    effective_overlap,
)
print("Input and stitched shapes:", data.shape, stitched.shape)
preview = [
    cp.asnumpy(sinogram[:half_count]),
    cp.asnumpy(opposing),
    cp.asnumpy(stitched[:, 0, :]),
]
lo, hi = np.percentile(preview[0][np.isfinite(preview[0])], [1, 99])
fig, axes = plt.subplots(3, 1, figsize=(12, 9), constrained_layout=True)
for ax, image, title in zip(
    axes,
    preview,
    (
        "First angular half",
        "Opposing half, detector columns reversed",
        "Stitched 180-degree sinogram",
    ),
):
    ax.imshow(image, cmap="gray", vmin=lo, vmax=hi, aspect="auto")
    ax.set(title=title, xlabel="Detector column (pixel)", ylabel="Projection index")
fig.suptitle("i13_dataset1: overlap and side from find_center_360")
plt.show()
plt.close(fig)
# Check full-stack stitching with the fixed reference overlap.
del stitched
clean_memory()
stitched_full = cp.asnumpy(sino_360_to_180(data, overlap=473.822265625, side="right"))
stitch_passed = (
    stitched_full.shape == (3000, 10, 5120)
    and stitched_full.dtype == np.float32
    and stitched_full.flags.c_contiguous
)
results.append(
    {
        "test": "test_sino_360_to_180_i13_dataset1",
        "overlap": 473.822265625,
        "side": "right",
        "shape": list(stitched_full.shape),
        "expected_shape": [3000, 10, 5120],
        "dtype": str(stitched_full.dtype),
        "c_contiguous": bool(stitched_full.flags.c_contiguous),
        "passed": bool(stitch_passed),
    }
)
print(json.dumps(results[-1], indent=2))
del stitched_full
del data, sinogram, opposing, preview, image
clean_memory()

## I13 DATASET3

Parameters: `{'ind': 1, 'win_width': 50, 'side': None, 'denoise': True, 'norm': True, 'use_overlap': True}`.
Check: `int(cor) == 2560`, `side == "left"` and
`int(overlap) == 438`. The integer comparison does not
require a fractional overlap to equal the integer reference exactly.


In [ ]:
data = load_transmission("i13_dataset3")
parameters = {
    "ind": 1,
    "win_width": 50,
    "side": None,
    "denoise": True,
    "norm": True,
    "use_overlap": True,
}
cor, overlap, side, overlap_position = find_center_360(data, **parameters)
passed = bool(int(cor) == 2560 and side == "left" and int(overlap) == 438)
results.append(
    {
        "test": "test_center_360_i13_dataset3",
        "dataset": "i13_dataset3",
        "parameters": parameters,
        "returned_center": float(cor),
        "expected_int_center": 2560,
        "overlap": float(overlap),
        "expected_int_overlap": 438,
        "side": side,
        "expected_side": "left",
        "overlap_position": float(overlap_position),
        "passed": passed,
    }
)
print(json.dumps(results[-1], indent=2))
row = parameters.get("ind", data.shape[1] // 2)
sinogram = data[:, row, :]
half_count = sinogram.shape[0] // 2 + 1
opposing = cp.fliplr(sinogram[-half_count:, :])
stitched = sino_360_to_180(data[:, row : row + 1, :], overlap=overlap, side=side)
effective_overlap = int(np.rint(overlap))
if effective_overlap % 2:
    effective_overlap += 1
print(
    "Estimated overlap:",
    float(overlap),
    "effective even-pixel overlap:",
    effective_overlap,
)
print("Input and stitched shapes:", data.shape, stitched.shape)
preview = [
    cp.asnumpy(sinogram[:half_count]),
    cp.asnumpy(opposing),
    cp.asnumpy(stitched[:, 0, :]),
]
lo, hi = np.percentile(preview[0][np.isfinite(preview[0])], [1, 99])
fig, axes = plt.subplots(3, 1, figsize=(12, 9), constrained_layout=True)
for ax, image, title in zip(
    axes,
    preview,
    (
        "First angular half",
        "Opposing half, detector columns reversed",
        "Stitched 180-degree sinogram",
    ),
):
    ax.imshow(image, cmap="gray", vmin=lo, vmax=hi, aspect="auto")
    ax.set(title=title, xlabel="Detector column (pixel)", ylabel="Projection index")
fig.suptitle("i13_dataset3: overlap and side from find_center_360")
plt.show()
plt.close(fig)
# Check full-stack stitching with the fixed reference overlap.
del stitched
clean_memory()
stitched_full = cp.asnumpy(sino_360_to_180(data, overlap=438.173828, side="left"))
stitch_passed = (
    stitched_full.shape == (3000, 3, 5120)
    and stitched_full.dtype == np.float32
    and stitched_full.flags.c_contiguous
)
results.append(
    {
        "test": "test_sino_360_to_180_i13_dataset3",
        "overlap": 438.173828,
        "side": "left",
        "shape": list(stitched_full.shape),
        "expected_shape": [3000, 3, 5120],
        "dtype": str(stitched_full.dtype),
        "c_contiguous": bool(stitched_full.flags.c_contiguous),
        "passed": bool(stitch_passed),
    }
)
print(json.dumps(results[-1], indent=2))
del stitched_full
del data, sinogram, opposing, preview, image
clean_memory()

## Final validation

Check integer centre, integer overlap and side for each dataset, followed by
full-stack stitched shape, float32 dtype and contiguous memory layout.


In [ ]:
check_results()